# เทรนโมเดลบนถนนไทย — fine-tune เทียบ เทรนใหม่ล้วน

ตอบคำถามหลักของโปรเจกต์: **โมเดลที่เทรนจากถนนจีน (CULane) ใช้กับถนนไทยได้แค่ไหน
และข้อมูลไทย 749 รูปช่วยได้เท่าไหร่**

สามการทดลอง วัดบนไทย test 191 รูปชุดเดียวกันทั้งหมด

| # | ทำอะไร | weights ตั้งต้น | เทรนด้วย |
| ---: | --- | --- | --- |
| 1 | zero-shot | CULane | ไม่เทรน (ทำไปแล้ว) |
| 2 | fine-tune | CULane | ไทย 749 รูป lr ต่ำ |
| 3 | เทรนใหม่ล้วน | ImageNet encoder | ไทย 749 รูป |

**โค้ดจริงอยู่ที่ `finetune_thai.py`** โน้ตบุ๊กนี้แค่ `import` มาเรียก ไม่มีสำเนาโค้ด
แก้สูตรให้ไปแก้ที่ `.py` ที่เดียว (แบบเดียวกับ `slope_metrics.py`)


In [ ]:
# ถ้ายังไม่มีของพวกนี้ ให้เอา # ออกแล้วรันหนึ่งครั้ง
# ต้องใส่ --user ไม่งั้นของหายเมื่อ container restart (CLAUDE.md)
# !pip install --user opencv-python numpy scipy segmentation-models-pytorch tqdm kornia


In [ ]:
import os
import sys
import csv
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.append('.')
# โค้ดเทรนทั้งหมดอยู่ใน finetune_thai.py — ที่นี่ import มาใช้ ไม่ก๊อป
from finetune_thai import run, THAI_ROOT, DEVICE
print(f"อุปกรณ์: {DEVICE.type.upper()}")
print(f"ข้อมูล: {THAI_ROOT}")
for s in ("train", "val", "test"):
    p = os.path.join(THAI_ROOT, f"{s}_list.csv")
    n = len(list(csv.DictReader(open(p)))) if os.path.exists(p) else 0
    print(f"  {s:5s} {n:4d} รูป  {'✓' if n else '✗ ไม่พบไฟล์'}")


## ตั้งค่า

`INIT_PTH` คือโมเดล CULane ที่จะใช้เป็นจุดตั้งต้นของ fine-tune

เลือก `best_model_resnet50_ssl_bs16_ep30.pth` เพราะ zero-shot บนไทยดีที่สุดทุกด้าน
(IoU 0.354 / มุมคลาด 1.90° / lane_recall 0.953)

**`LR_FINETUNE` ต้องต่ำกว่า `LR_SCRATCH` สิบเท่า** ถ้าใช้ lr เท่ากัน fine-tune จะลืม
feature จาก CULane จนกลายเป็นเทรนใหม่กลายๆ แล้วข้อ 2 กับข้อ 3 จะแยกกันไม่ออก


In [ ]:
# ==================== แก้ตรงนี้อย่างเดียว ====================
INIT_PTH    = "models/best_model_resnet50_ssl_bs16_ep30.pth"
ENCODER     = "resnet50"      # ต้องตรงกับ encoder ของ INIT_PTH
EPOCHS      = 30
BATCH_SIZE  = 16
LR_FINETUNE = 1e-5            # ต่ำ เพื่อไม่ให้ลืมของเดิม
LR_SCRATCH  = 1e-4            # เท่าตอนเทรน CULane รอบแรก
# ================================================================

assert os.path.exists(INIT_PTH), f"ไม่พบ {INIT_PTH}"
print(f"จะ fine-tune จาก: {os.path.basename(INIT_PTH)}")
print(f"encoder={ENCODER} | epochs={EPOCHS} | bs={BATCH_SIZE}")
print(f"lr: finetune={LR_FINETUNE} / scratch={LR_SCRATCH}")


## การทดลองที่ 2 — fine-tune

เริ่มจาก weights ของ CULane แล้วเทรนต่อด้วยไทย 749 รูป

ใช้เวลาราว 20 นาที (30 epoch × ~35 วินาที)

**ตัวตรวจสอบ:** ผลต้องดีกว่า zero-shot (IoU 0.354 / มุมคลาด 1.90°)
ถ้าแย่กว่า แปลว่า lr สูงไปหรือโหลดข้อมูลผิด — หยุดตรวจก่อน อย่าอ่านผลต่อ


In [ ]:
res_ft = run(mode="finetune", init=INIT_PTH, encoder=ENCODER,
             epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR_FINETUNE,
             tag="finetune_ssl")


## การทดลองที่ 3 — เทรนใหม่ล้วน

เริ่มจาก encoder ที่ pretrain ImageNet มาเท่านั้น ไม่เคยเห็นเส้นเลนมาก่อน
เทรนด้วยไทย 749 รูปอย่างเดียว

**คาดว่าจะ overfit หนัก** เพราะเคยวัดแล้วบน CULane subset 2,000 รูป: val F1 ตันที่ 0.61
ตั้งแต่ epoch 13 ขณะ train F1 ไต่ถึง 0.886 — ไทยมีแค่ 749 รูป (37% ของ 2,000)

**ตัวตรวจสอบ:** ต้องเห็นช่องว่าง train F1 กับ val F1 กว้างขึ้นเรื่อยๆ
ถ้าไม่เห็น แปลว่าอ่านข้อมูลผิด


In [ ]:
res_sc = run(mode="scratch", encoder=ENCODER,
             epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR_SCRATCH,
             tag="scratch_ssl")


## เทียบสามทาง

ตารางนี้คือผลลัพธ์หลักของโปรเจกต์ — ตอบว่าต้องใช้ข้อมูลไทยแค่ไหนถึงจะพอ


In [ ]:
import numpy as np

FILES = [("1. zero-shot (CULane ล้วน)", "results/thai/zeroshot_unet_resnet50_ssl_bs16_ep30.csv"),
         ("2. fine-tune (CULane + ไทย)", "results/thai/finetune_ssl.csv"),
         ("3. เทรนใหม่ล้วน (ไทยอย่างเดียว)", "results/thai/scratch_ssl.csv")]

def load(p, scope="ALL"):
    if not os.path.exists(p):
        return None
    r = [x for x in csv.DictReader(open(p)) if x["scope"] == scope]
    return r[0] if r else None

print("ผลบนไทย test 191 รูป (ชุดเดียวกันทั้งสามแถว)\n")
print(f"{'วิธี':34s}{'IoU':>8s}{'F1':>8s}{'มุมคลาด':>10s}{'recall':>9s}")
print("-" * 69)
base = None
for name, p in FILES:
    r = load(p)
    if r is None:
        print(f"  {name:32s}{'ยังไม่ได้รัน':>35s}")
        continue
    if base is None:
        base = r
    print(f"  {name:32s}{float(r['iou']):8.3f}{float(r['f1']):8.3f}"
          f"{float(r['median_err']):9.2f}°{float(r['lane_recall']):9.3f}")

# เทียบกับ zero-shot ว่าขยับไปเท่าไหร่
if base is not None:
    print(f"\nเปลี่ยนจาก zero-shot")
    for name, p in FILES[1:]:
        r = load(p)
        if r is None:
            continue
        di = (float(r["iou"]) - float(base["iou"])) / float(base["iou"]) * 100
        da = (float(r["median_err"]) - float(base["median_err"])) / float(base["median_err"]) * 100
        print(f"  {name:32s} IoU {di:+6.1f}%   มุมคลาด {da:+6.1f}%")

print(f"\nแยกตามรูปแบบถนน")
for name, p in FILES:
    if not os.path.exists(p):
        continue
    print(f"\n  {name}")
    print(f"    {'ถนน':8s}{'รูป':>6s}{'IoU':>8s}{'F1':>8s}{'มุมคลาด':>10s}{'recall':>9s}")
    for s in ("normal", "curve", "night"):
        r = load(p, s)
        if r is None:
            continue
        print(f"      {s:6s}{r['n_img']:>6s}{float(r['iou']):8.3f}{float(r['f1']):8.3f}"
              f"{float(r['median_err']):9.2f}°{float(r['lane_recall']):9.3f}")


## ข้อควรระวังตอนอ่านผล

**test ไทยเอียงหนัก** `normal` 157 จาก 191 รูป (82%) ส่วน `curve` เหลือแค่ 17 รูป
ตัวเลขรวมจึงถูก `normal` ครอบงำ — ต้องดูตารางแยกหมวดเสมอ

**ทางโค้ง 17 รูปแกว่งได้มาก** zero-shot ได้ lane_recall 0.613 ที่นั่น
อย่าสรุปแน่นจากตัวเลขเดียว

**val ไทยมีแค่ 97 รูป** ใช้เลือก checkpoint ได้ แต่อย่ารายงานเป็นผล

**สัดส่วนภาพต่างกัน** ไทยต้นฉบับ 1920×1080 (16:9) CULane 1640×590 (16:5.76)
ทั้งคู่ถูกบีบเป็น 800×448 คนละอัตรา ทรงเส้นเลนจึงต่างกัน เป็นตัวแปรปนที่แยกออกไม่ได้

**ทั้งสามการทดลองใช้ `normalize=False`** ตรงกับที่ตรวจไว้ว่า weights ชุดนี้เทรนมาแบบ
input `[0,1]` ถ้าใส่ผิดตัวเลขพังเงียบโดยไม่ error
